# Value iteration and policy iteration

**Learning goals:** implement both algorithms for one finite MDP, compare their update patterns, and verify the greedy fixed point.

**Predict first:** which method uses fewer outer iterations, and which method does more work inside an iteration?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

P = np.array([
    [[0.85, 0.15, 0.00], [0.10, 0.75, 0.15], [0.00, 0.20, 0.80]],
    [[0.20, 0.70, 0.10], [0.00, 0.25, 0.75], [0.10, 0.10, 0.80]],
])  # action, state, next state
R = np.array([[0.0, 0.5], [0.3, 1.0], [1.0, 0.2]])
gamma = 0.93

def q_values(value):
    return R + gamma * np.einsum("asj,j->sa", P, value)

def value_iteration(tol=1e-10):
    value, residuals = np.zeros(3), []
    for _ in range(10_000):
        updated = q_values(value).max(axis=1)
        residuals.append(np.max(np.abs(updated - value)))
        value = updated
        if residuals[-1] < tol:
            break
    return value, q_values(value).argmax(axis=1), np.asarray(residuals)

def policy_iteration():
    policy, history = np.zeros(3, dtype=int), []
    while True:
        P_pi = P[policy, np.arange(3)]
        r_pi = R[np.arange(3), policy]
        value = np.linalg.solve(np.eye(3) - gamma * P_pi, r_pi)
        improved = q_values(value).argmax(axis=1)
        history.append(value.copy())
        if np.array_equal(improved, policy):
            return value, policy, np.asarray(history)
        policy = improved

v_vi, pi_vi, residuals = value_iteration()
v_pi, pi_pi, pi_history = policy_iteration()
print("Value-iteration policy:", pi_vi)
print("Policy-iteration policy:", pi_pi)

In [ ]:
plt.figure(figsize=(6, 3))
plt.semilogy(residuals)
plt.xlabel("Value-iteration update")
plt.ylabel("Bellman residual")
plt.grid(alpha=0.25)
plt.show()

**Experiment:** set `gamma` to 0.5 and rerun. Which policy changes, if any? Explain why a faster contraction need not imply a different optimum.

In [ ]:
assert np.allclose(P.sum(axis=2), 1.0)
assert np.allclose(v_vi, v_pi, atol=1e-8)
assert np.array_equal(pi_vi, pi_pi)
assert np.max(np.abs(q_values(v_vi).max(axis=1) - v_vi)) < 1e-8
print("Checks passed.")